# LangGraph State Machines

**WatSPEED Agentic AI prep — Week 4 - orchestration**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## LangGraph: when a loop isn't enough

The ReAct loop in notebook 03 is one shape: *model -> tool -> model*. Real workflows
branch, retry, and pause for a human. LangGraph models that as an explicit
**state machine**: typed state, nodes that transform it, edges that decide what runs next.

If you've written SAS macros with `%IF` branching, this is the same idea with the
control flow made into data you can inspect.

We build the machine ourselves first, then show the LangGraph syntax.

### The "Why": Why State Machines replace Spaghetti Code

> **The Legacy Friction:** When analysts try to build complex logic, they often end up with massive, nested `IF/THEN/ELSE` blocks in a single SAS script or Python function. This 'spaghetti code' is impossible to audit, hard to debug, and dangerous to modify.
>
> **The RAP Value Proposition:** LangGraph forces you to organize complex logic into a **State Machine** (a DAG). Every step is a distinct node, and data flows explicitly between them. This allows you to easily audit execution paths, inject 'Human-in-the-Loop' pauses (where a human must click 'Approve' before continuing), and mathematically guarantee that your agent won't get stuck in an infinite loop.


In [2]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class AnalysisState:
    """Typed state, threaded through every node. LangGraph calls this the schema."""
    question: str
    hypothesis: str = ""
    result: dict = field(default_factory=dict)
    critique: str = ""
    attempts: int = 0
    done: bool = False
    log: list = field(default_factory=list)

    def note(self, msg: str) -> None:
        self.log.append(msg)

### Nodes: plain functions, state in -> state out

### The Deterministic Cage: LangGraph as the Tracks

> **The Fear of Non-Determinism (Part 2):** If you give an LLM 50 tools, it might wander off and execute them in a chaotic, unpredictable order.
>
> **The Solution:** LangGraph forces the LLM onto a strict set of train tracks (a State Machine). You mathematically define the rules (e.g., `"You can only use the Regression tool AFTER the Clean Data tool succeeds"`). This cages the LLM's non-determinism, ensuring it only does productive, auditable work in the exact order you permit.


In [3]:
def form_hypothesis(s: AnalysisState) -> AnalysisState:
    s.hypothesis = "AI trust differs by age group"
    s.note(f"hypothesis: {s.hypothesis}")
    return s

def run_test(s: AnalysisState) -> AnalysisState:
    s.attempts += 1
    # First attempt 'forgets' survey weights - deliberately, so review can catch it.
    s.result = ({"chi2": 12.41, "p": 0.006, "weighted": s.attempts > 1, "n": 2041})
    s.note(f"attempt {s.attempts}: p={s.result['p']}, weighted={s.result['weighted']}")
    return s

def review(s: AnalysisState) -> AnalysisState:
    if not s.result.get("weighted"):
        s.critique = "Estimates are unweighted; survey weights are required."
    else:
        s.critique = ""
        s.done = True
    s.note(f"review: {s.critique or 'passed'}")
    return s

### Edges: the routing function

This is the piece that makes it a graph rather than a script. `route` reads state and
names the next node. Note the attempt cap — a cyclic graph without one is an infinite bill.

In [4]:
def route(s: AnalysisState) -> str:
    if s.done:
        return "END"
    if s.critique and s.attempts < 3:
        return "run_test"          # cycle back and retry
    if s.attempts >= 3:
        return "END"
    return "review"

NODES: dict[str, Callable] = {"form_hypothesis": form_hypothesis, "run_test": run_test, "review": review}

def run_graph(state: AnalysisState, start: str = "form_hypothesis") -> AnalysisState:
    node, step = start, 0
    order = {"form_hypothesis": "run_test", "run_test": "review"}
    while node != "END" and step < 12:
        step += 1
        state = NODES[node](state)
        node = route(state) if node == "review" else order[node]
        trace(step, node if node != "END" else "END", state.log[-1])
    return state

banner("Graph execution")
final = run_graph(AnalysisState(question="Does AI trust vary by age?"))
print(f"\nfinished after {final.attempts} attempts, done={final.done}")
show("final result", final.result)


Graph execution
  [ 1] run_test     | hypothesis: AI trust differs by age group
  [ 2] review       | attempt 1: p=0.006, weighted=False
  [ 3] run_test     | review: Estimates are unweighted; survey weights are required.
  [ 4] review       | attempt 2: p=0.006, weighted=True
  [ 5] END          | review: passed

finished after 2 attempts, done=True
final result:
  {
    "chi2": 12.41,
    "p": 0.006,
    "weighted": true,
    "n": 2041
  }


### Human in the loop

The syllabus calls for approval gates. In a state machine that's just a node that
returns without advancing — the graph checkpoints, and you resume it later with a decision.

In [5]:
@dataclass
class Gate:
    pending: dict | None = None

    def request(self, action: str, detail: dict) -> str:
        self.pending = {"action": action, "detail": detail}
        return f"PAUSED - awaiting approval to {action}"

    def resume(self, approved: bool) -> str:
        act = self.pending["action"]
        self.pending = None
        return f"{'EXECUTED' if approved else 'REJECTED'}: {act}"

gate = Gate()
print(gate.request("publish findings to the shared report", final.result))
print(gate.resume(approved=True))

PAUSED - awaiting approval to publish findings to the shared report
EXECUTED: publish findings to the shared report


### The LangGraph syntax

```python
from langgraph.graph import StateGraph, END

g = StateGraph(AnalysisState)
g.add_node("hypothesis", form_hypothesis)
g.add_node("test", run_test)
g.add_node("review", review)

g.set_entry_point("hypothesis")
g.add_edge("hypothesis", "test")
g.add_edge("test", "review")
g.add_conditional_edges("review", route, {"run_test": "test", "END": END})

app = g.compile(checkpointer=MemorySaver())   # checkpointer = the pause/resume above
app.invoke({"question": "Does AI trust vary by age?"})
```

`add_node`, `add_edge`, `add_conditional_edges` — the three calls map exactly to
`NODES`, `order` and `route`.

### Guided Exercise: Building the StateGraph

Let's explicitly build the StateGraph (The Train Tracks). This forces the LLM to follow a strict pipeline, completely eliminating spaghetti `IF/THEN/ELSE` code.

In [ ]:
from typing import TypedDict, Annotated
import operator

# STEP 1: Define the State 
# Think of this exactly like the variables stored in the SAS Program Data Vector (PDV) during execution.
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    data_is_clean: bool

# STEP 2: Define Nodes (Python functions taking State -> State)
def clean_data_node(state: AgentState):
    print("-> Executing Data Cleaning...")
    return {"data_is_clean": True, "messages": ["Data cleaned."]}

def run_regression_node(state: AgentState):
    print("-> Running Regression Model...")
    return {"messages": ["Regression successful."]}

# STEP 3: Define the strict routing logic
def routing_logic(state: AgentState):
    if state.get("data_is_clean", False):
        return "run_regression"
    else:
        return "clean_data"

print("StateGraph logic defined! This guarantees the agent can NEVER run a regression on dirty data.")